# KDIC Structured Hybrid + Parent-Child 검색 평가기

업무 분류와 업무 필터 없이 전체 청크를 대상으로 다음 흐름을 평가합니다.

`Structured BGE-M3 Dense Top-20 + BM25-Nori Discard Top-20 → Weighted RRF → Child Top-10 → Parent 내 인접 청크 ±1 확장`

표준 검색 지표는 **확장 전 Child 순위**로 계산하고, Parent-Child 확장 효과는 별도 지표로 저장합니다.

## 준비 파일

1. `KDIC_Structured_Hybrid_ParentChild_평가기.zip`
2. `KDIC_output.zip`
3. `test_dataset_4.1.xlsx`

원본 평가데이터셋은 읽기만 하며 수정하지 않습니다.


In [ ]:
%pip -q install "numpy==2.0.2" "pandas==2.2.2" "openpyxl>=3.1,<4" "JPype1>=1.5,<2"

import getpass
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

WORK_ROOT=Path('/content/kdic_structured_hybrid_parent_child_evaluation')
EVALUATOR_ROOT=WORK_ROOT/'evaluator'
DRY_ROOT=WORK_ROOT/'dry_run'
RESULT_ROOT=WORK_ROOT/'results'
WORK_ROOT.mkdir(parents=True,exist_ok=True)
print('환경 준비 완료:',WORK_ROOT)
print('Java:',shutil.which('java'))
if shutil.which('java') is None:
    raise RuntimeError('Java를 찾지 못했습니다. Colab 기본 런타임에서 다시 실행해 주세요.')


## 1. 파일 3개 업로드

준비 파일 3개를 한 번에 선택합니다.


In [ ]:
from google.colab import files
uploaded=files.upload()
for filename,content in uploaded.items():
    target=WORK_ROOT/filename
    target.write_bytes(content)
    print(f'업로드: {target.name} ({target.stat().st_size:,} bytes)')


In [ ]:
dataset_candidates=list(WORK_ROOT.glob('*.xlsx'))
kdic_candidates=[p for p in WORK_ROOT.glob('*.zip') if p.name=='KDIC_output.zip' or p.name.startswith('KDIC_output (')]
evaluator_candidates=[p for p in WORK_ROOT.glob('*.zip') if 'Structured_Hybrid_ParentChild' in p.name and '평가기' in p.name]

if not dataset_candidates:
    raise RuntimeError('평가데이터셋 XLSX를 찾을 수 없습니다.')
if not kdic_candidates:
    raise RuntimeError('KDIC_output.zip을 찾을 수 없습니다.')
if not evaluator_candidates:
    raise RuntimeError('Structured Hybrid ParentChild 평가기 ZIP을 찾을 수 없습니다.')

choose_latest=lambda items:max(items,key=lambda p:p.stat().st_mtime)
DATASET_PATH=choose_latest(dataset_candidates)
KDIC_ZIP_PATH=choose_latest(kdic_candidates)
EVALUATOR_ZIP_PATH=choose_latest(evaluator_candidates)

if EVALUATOR_ROOT.exists():
    shutil.rmtree(EVALUATOR_ROOT)
EVALUATOR_ROOT.mkdir(parents=True)
with zipfile.ZipFile(EVALUATOR_ZIP_PATH) as archive:
    archive.extractall(EVALUATOR_ROOT)
scripts=list(EVALUATOR_ROOT.rglob('evaluate_structured_hybrid_parent_child.py'))
if len(scripts)!=1:
    raise RuntimeError(f'평가 스크립트를 하나로 결정할 수 없습니다: {scripts}')
EVALUATOR_SCRIPT=scripts[0]

print('평가데이터셋:',DATASET_PATH.name)
print('검색 데이터:',KDIC_ZIP_PATH.name)
print('평가기:',EVALUATOR_ZIP_PATH.name)


## 2. 실험 설정

먼저 기본값으로 실행하고, 후속 실험에서 한 번에 한 항목만 변경하세요.


In [ ]:
DENSE_WEIGHT=0.85
NORI_WEIGHT=0.15
RRF_CONSTANT=10
CANDIDATE_K=20
FINAL_K=10

SEED_CHILD_K=5
NEIGHBOR_WINDOW=1
MAX_PARENTS=3
MAX_CHUNKS_PER_PARENT=3
MAX_CONTEXT_CHUNKS=10

BM25_K1=1.5
BM25_B=0.75
LUCENE_VERSION='9.12.2'

print({
    '업무 필터':'미적용',
    'Structured 필드':['title','section_title','heading_path','content'],
    'Dense:Nori':f'{DENSE_WEIGHT}:{NORI_WEIGHT}',
    'RRF c':RRF_CONSTANT,
    '각 검색기 후보':CANDIDATE_K,
    '최종 Child':FINAL_K,
    '확장 seed Child':SEED_CHILD_K,
    '인접 범위':f'±{NEIGHBOR_WINDOW}',
    '최대 Parent':MAX_PARENTS,
    'Parent당 최대 청크':MAX_CHUNKS_PER_PARENT,
    '최대 컨텍스트 청크':MAX_CONTEXT_CHUNKS,
})


## 3. Dry-run

API를 호출하기 전에 최신 평가대상 질문, 427개 청크, Gold와 Parent 메타데이터를 검사합니다.


In [ ]:
if DRY_ROOT.exists():
    shutil.rmtree(DRY_ROOT)
cmd=[
    sys.executable,str(EVALUATOR_SCRIPT),
    '--dataset',str(DATASET_PATH),
    '--sheet-name','Sheet1',
    '--kdic-zip',str(KDIC_ZIP_PATH),
    '--output-dir',str(DRY_ROOT),
    '--candidate-k',str(CANDIDATE_K),
    '--final-k',str(FINAL_K),
    '--dense-weight',str(DENSE_WEIGHT),
    '--nori-weight',str(NORI_WEIGHT),
    '--rrf-constant',str(RRF_CONSTANT),
    '--seed-child-k',str(SEED_CHILD_K),
    '--neighbor-window',str(NEIGHBOR_WINDOW),
    '--max-parents',str(MAX_PARENTS),
    '--max-chunks-per-parent',str(MAX_CHUNKS_PER_PARENT),
    '--max-context-chunks',str(MAX_CONTEXT_CHUNKS),
    '--dry-run',
]
subprocess.run(cmd,cwd=EVALUATOR_ROOT,check=True)
dry_report=json.loads((DRY_ROOT/'dry_run_report.json').read_text(encoding='utf-8'))
display(pd.DataFrame([dry_report]))


## 4. HCX API 키

Colab Secrets의 `HCX_API_KEY`를 우선 사용하며 없을 때만 보안 입력창을 표시합니다. 키 앞에 `Bearer`를 붙이지 않습니다.


In [ ]:
from google.colab import userdata

try:
    HCX_API_KEY=(userdata.get('HCX_API_KEY') or '').strip()
except Exception:
    HCX_API_KEY=''

if HCX_API_KEY:
    print('Colab Secrets의 HCX_API_KEY를 불러왔습니다.')
else:
    HCX_API_KEY=getpass.getpass('HCX API 키를 입력하세요: ').strip()

if not HCX_API_KEY:
    raise RuntimeError('API 키가 비어 있습니다.')
if HCX_API_KEY.lower().startswith('bearer '):
    raise RuntimeError("키 앞에 'Bearer '를 붙이지 마세요.")
os.environ['HCX_API_KEY']=HCX_API_KEY
print('현재 Colab 세션에 API 키를 등록했습니다.')


## 5. 전체 평가 실행

최초 실행은 Structured 문서 임베딩 427개 생성 때문에 시간이 걸립니다. 같은 결과 폴더로 재실행하면 캐시를 사용합니다.


In [ ]:
RESULT_ROOT.mkdir(parents=True,exist_ok=True)
cmd=[
    sys.executable,str(EVALUATOR_SCRIPT),
    '--dataset',str(DATASET_PATH),
    '--sheet-name','Sheet1',
    '--kdic-zip',str(KDIC_ZIP_PATH),
    '--output-dir',str(RESULT_ROOT),
    '--candidate-k',str(CANDIDATE_K),
    '--final-k',str(FINAL_K),
    '--dense-weight',str(DENSE_WEIGHT),
    '--nori-weight',str(NORI_WEIGHT),
    '--rrf-constant',str(RRF_CONSTANT),
    '--seed-child-k',str(SEED_CHILD_K),
    '--neighbor-window',str(NEIGHBOR_WINDOW),
    '--max-parents',str(MAX_PARENTS),
    '--max-chunks-per-parent',str(MAX_CHUNKS_PER_PARENT),
    '--max-context-chunks',str(MAX_CONTEXT_CHUNKS),
    '--bm25-k1',str(BM25_K1),
    '--bm25-b',str(BM25_B),
    '--lucene-version',LUCENE_VERSION,
]
subprocess.run(cmd,cwd=EVALUATOR_ROOT,check=True)


## 6. 결과 확인


In [ ]:
summary=json.loads((RESULT_ROOT/'summary.json').read_text(encoding='utf-8'))
overall=pd.DataFrame([summary['overall']])
display(overall)

questions=pd.read_csv(RESULT_ROOT/'question_results.csv')
display(questions[[
    'evaluation_id','question','hit_at_3','recall_at_5',
    'child_seed_recall','expanded_gold_recall','expansion_recall_gain',
    'parent_hit_at_3','expanded_context_chunk_count','latency_ms'
]].head(20))

improved=questions[questions['expansion_recall_gain']>0][[
    'evaluation_id','question','seed_child_ids','expanded_context_chunk_ids',
    'child_seed_recall','expanded_gold_recall','expansion_recall_gain'
]]
print('Parent-Child 확장으로 Gold Recall이 증가한 질문:',len(improved))
display(improved.head(30))


## 7. 결과 ZIP 다운로드

기존 KDIC 검색평가 비교 대시보드에는 표준 검색 지표가 그대로 들어가며, Parent-Child 추가 열도 CSV에 보존됩니다.


In [ ]:
RESULT_ZIP=Path('/content/KDIC_Structured_Hybrid_ParentChild_업무필터없음_평가결과.zip')
if RESULT_ZIP.exists():
    RESULT_ZIP.unlink()
with zipfile.ZipFile(RESULT_ZIP,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULT_ROOT.rglob('*')):
        if path.is_file():
            archive.write(path,path.relative_to(RESULT_ROOT).as_posix())
print('결과 ZIP:',RESULT_ZIP,RESULT_ZIP.stat().st_size)
files.download(str(RESULT_ZIP))
